# ScanForge backend on Colab

Runs the server side for testing: the FastAPI backend and a public ngrok address the phone
app talks to. Scans are processed by 3D services, so **no GPU is needed**: a normal (CPU) runtime works.

| Option in the app | Service | Secret to add (🔑 in the left sidebar) | Cost |
|---|---|---|---|
| Basic | fal.ai TRELLIS (open source) | `FAL_KEY` | ~$0.02 |
| Standard | Tripo (HD texture) | `TRIPO_API_KEY` | ~$0.40 |
| Premium | Rodin Gen-2.5 (fal.ai, 4K textures) | `FAL_KEY` | $1.20 |

The app only offers options whose key is set. Also add `NGROK_TOKEN`.

Scans are paid before processing. Sideloaded test APKs can't use Google Play, so this
notebook turns on test payments (`ALLOW_DEV_BILLING=1`): the app skips payment. Never set that in production.

Run steps 1, 2 and 4, and keep this tab open. Steps 3 and 5 are only for the optional
self-hosted GPU pipeline.

## 1. Settings

In [ ]:
NGROK_DOMAIN = "chamber-barber-dole.ngrok-free.dev"   # must match the APK's API base

import getpass, os
try:
    from google.colab import userdata
    def secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
except ImportError:
    secret = lambda name: None

NGROK_TOKEN = secret("NGROK_TOKEN") or getpass.getpass("ngrok auth token: ").strip()
for name in ("TRIPO_API_KEY", "FAL_KEY", "KIRI_API_KEY"):
    value = secret(name)
    if value:
        os.environ[name] = value
print("services configured:", [n for n in ("TRIPO_API_KEY", "FAL_KEY", "KIRI_API_KEY") if os.environ.get(n)] or "none")

## 2. Get the code and backend dependencies (~1 min)

In [ ]:
# Uses Colab's existing package versions (no pins), so nothing Colab relies on gets downgraded.
!rm -rf /content/scanforge && git clone -q --depth 1 https://github.com/infinityinketan-sketch/scanforge.git /content/scanforge
!pip -q install fastapi "uvicorn[standard]" python-multipart httpx google-api-python-client google-auth pyngrok trimesh pillow numpy

import importlib
for m in ["fastapi", "uvicorn", "multipart", "httpx", "googleapiclient", "pyngrok", "trimesh", "PIL"]:
    importlib.import_module(m)
import os; assert os.path.exists("/content/scanforge/backend/main.py"), "clone failed"
print("✅ backend ready (a red 'pip dependency resolver' note above is harmless)")


## 3. (Optional, GPU runtime only) Install COLMAP with CUDA (~2–3 min, once per session)

Uses COLMAP's official CUDA build from PyPI (`pycolmap-cuda12`) in its own environment
(`/content/recon`), so Colab's packages aren't touched.

In [ ]:
%%bash
set -e
nvidia-smi -L || { echo "No GPU: Runtime > Change runtime type > T4 GPU, then rerun"; exit 1; }
if ! /content/recon/bin/python -c "import pycolmap, open3d" 2>/dev/null; then
  rm -rf /content/recon
  pip -q install uv
  uv venv -q -p "$(which python3)" /content/recon
  uv pip install -q -p /content/recon/bin/python "pycolmap-cuda12==4.2.1" open3d trimesh requests numpy
fi
/content/recon/bin/python -c "import pycolmap; print('pycolmap', pycolmap.__version__, '| CUDA:', pycolmap.has_cuda); assert pycolmap.has_cuda, 'no CUDA build'"
echo "✅ COLMAP ready"

## 4. Start the backend and the public ngrok address

In [ ]:
import os, secrets, subprocess, sys, time, requests
from pyngrok import ngrok

API_BASE = f"https://{NGROK_DOMAIN}"
os.environ["DATA_DIR"] = "/content/data"
os.environ["API_BASE"] = API_BASE
os.environ["ALLOW_DEV_BILLING"] = "1"
os.environ["PROCESSING_TIMEOUT"] = "5400"   # a free T4 can need 20–40 min per scan   # test only: lets the helper below unlock exports
os.environ.setdefault("SECRET_KEY", secrets.token_hex(16))   # stable for this session
SECRET = os.environ["SECRET_KEY"]

# Restart cleanly if this cell is rerun
subprocess.run("pkill -f 'uvicorn main:app'", shell=True)
ngrok.kill()
time.sleep(1)

log_file = open("/content/backend.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/scanforge/backend", env=os.environ.copy(),
    stdout=log_file, stderr=subprocess.STDOUT,
)
for _ in range(30):
    time.sleep(1)
    try:
        if requests.get("http://localhost:8000/healthz", timeout=2).ok:
            break
    except requests.RequestException:
        pass
else:
    print(open("/content/backend.log").read()[-3000:])
    raise RuntimeError("Backend did not start; log above")

ngrok.set_auth_token(NGROK_TOKEN)
# The domain allows one tunnel at a time. A tunnel from an earlier Colab session can hold it
# for a minute or two after that session ends, so retry before giving up.
from pyngrok.exception import PyngrokNgrokHTTPError
for attempt in range(12):
    try:
        ngrok.connect(8000, domain=NGROK_DOMAIN)
        break
    except PyngrokNgrokHTTPError as e:
        if "ERR_NGROK_334" not in str(e):
            raise
        print(f"ngrok address still held by an older session, retrying ({attempt + 1}/12)…")
        time.sleep(15)
else:
    raise RuntimeError(
        "Your ngrok address is still in use by another session. Close other Colab tabs running this notebook "
        "(Runtime > Manage sessions > Terminate), or stop it at https://dashboard.ngrok.com/agents, then rerun this cell.")
r = requests.get(f"{API_BASE}/healthz", headers={"ngrok-skip-browser-warning": "1"}, timeout=15)
print("public health check:", r.status_code, r.text)
print("✅ Backend live at", API_BASE)
print("Options offered to the app:", [t["name"] for t in requests.get("http://localhost:8000/tiers").json()["tiers"]] or "none (add API keys in step 1)")

## 5. (Optional, GPU runtime only) Self-hosted processing (leave running)

Picks up every uploaded scan, runs COLMAP + meshing on the GPU, and posts the GLB/STL back.
The phone app updates by itself when a scan finishes. Stop this cell to stop processing.

In [ ]:
import json
sys.path.insert(0, "/content/scanforge/backend")
import db   # same SQLite file as the server (DATA_DIR above)

LOCAL = "http://localhost:8000"
RUNNER = "import sys, json; sys.path.insert(0, '/content/scanforge/pipeline'); import handler; " \
         "print(json.dumps(handler.handler(json.loads(sys.argv[1]))['ok']))"
env = dict(os.environ, COLMAP_BACKEND="pycolmap", JOB_DIR="/content/work",
           KEEP_DIR="/content/data/points")   # keeps 3D points so a scan can be re-meshed quickly

def run_job(job_id):
    m = requests.get(f"{LOCAL}/jobs/{job_id}/manifest", params={"key": SECRET}, timeout=30).json()
    # Fetch photos and post results over localhost instead of through ngrok.
    swap = lambda u: u.replace(API_BASE, LOCAL, 1)
    event = {"input": {"job_id": job_id,
                       "photo_urls": [swap(u) for u in m["photo_urls"]],
                       "result_url": swap(m["result_url"])}}
    print(f"▶ {job_id}: {len(event['input']['photo_urls'])} photos")
    started = time.time()
    lines = []
    p = subprocess.Popen(["/content/recon/bin/python", "-u", "-c", RUNNER, json.dumps(event)],
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f"/content/worker-{job_id}.log", "w") as log:
        for line in p.stdout:
            log.write(line); log.flush()
            lines.append(line)
            if line.startswith("+ ") or "rror" in line:   # pipeline steps and errors; full log in the file
                print(f"  [{(time.time() - started) / 60:4.1f} min] {line.rstrip()[:200]}")
    p.wait()
    job = db.get_job(job_id)
    if job["status"] == "processing":   # worker crashed before reporting back
        tail = [l.strip() for l in lines if l.strip()][-1:] or ["worker crashed"]
        db.update_job(job_id, status="failed", error=tail[0][:500], finished=time.time())
    print(f"  {job_id}: {db.get_job(job_id)['status']}  {db.get_job(job_id)['error'] or ''}")

def ensure_backend():
    """Restart the API server if it died (e.g. Colab ran short of memory during a big job)."""
    global server
    try:
        if requests.get(f"{LOCAL}/healthz", timeout=5).ok:
            return
    except requests.RequestException:
        pass
    print("⚠ backend was down, restarting it")
    server = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd="/content/scanforge/backend", env=os.environ.copy(),
        stdout=open("/content/backend.log", "a"), stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        time.sleep(1)
        try:
            if requests.get(f"{LOCAL}/healthz", timeout=2).ok:
                return
        except requests.RequestException:
            pass
    raise RuntimeError("backend won't start; see /content/backend.log")

print("Waiting for scans…")
while True:
    ensure_backend()
    for row in db._conn().execute("SELECT id FROM jobs WHERE status='queued' ORDER BY created_at").fetchall():
        if db.claim_status(row["id"], ("queued",), "processing", started=time.time()):
            try:
                ensure_backend()
                run_job(row["id"])
            except Exception as e:
                db.update_job(row["id"], status="failed", error=f"worker: {e}"[:500], finished=time.time())
                print("  failed:", e)
    time.sleep(5)

## Re-mesh a scan (1–3 min instead of an hour)

Stop step 5 first. Rebuilds the final model from a scan's saved 3D points using the latest
code from GitHub, and updates the scan in the app. Leave `JOB` blank for the most recent scan.
To share a scan's points for debugging, download `/content/data/points/<job id>/fused.ply`
from the Files panel.


In [ ]:
import glob
JOB = ""   # leave blank for the most recent scan with saved points

if not JOB:
    saved = sorted(glob.glob("/content/data/points/*/fused.ply"), key=os.path.getmtime)
    JOB = saved[-1].split("/")[-2] if saved else ""
assert JOB, "No saved points yet: scans keep their points from this notebook version on."

subprocess.run(["git", "-C", "/content/scanforge", "pull", "-q"])   # newest mesh code
REMESH = r"""
import os, sys, requests
sys.path.insert(0, "/content/scanforge/pipeline"); import handler
job, result_url = sys.argv[1], sys.argv[2]
out = f"/content/remesh/{job}"; os.makedirs(out, exist_ok=True)
log = []
try:
    handler._mesh(f"/content/data/points/{job}/fused.ply", f"{out}/model.stl",
                  f"{out}/model.glb", f"{out}/preview.glb", log)
    print("+ " + " | ".join(log))
    names = {"stl": "model.stl", "glb": "model.glb", "preview": "preview.glb"}
    files = {k: (n, open(f"{out}/{n}", "rb")) for k, n in names.items()}
    requests.post(result_url, files=files, data={"log": "\n".join(log)}, timeout=600).raise_for_status()
    print("✅ new model sent to the app")
except Exception as e:
    print("error:", e, "|", " | ".join(log))
"""
ensure_backend()
m = requests.get(f"{LOCAL}/jobs/{JOB}/manifest", params={"key": SECRET}, timeout=30).json()
print("re-meshing", JOB, "…")
r = subprocess.run(["/content/recon/bin/python", "-c", REMESH, JOB,
                    m["result_url"].replace(API_BASE, LOCAL, 1)], capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])


## Testing helpers (stop step 5 first, run, then restart step 5)

The APK is sideloaded, so Google Play can't sell the export unlock. This marks the most recent finished scan as paid
so you can test GLB/STL downloads.

In [ ]:
row = db._conn().execute("SELECT id FROM jobs WHERE status='done' ORDER BY finished DESC LIMIT 1").fetchone()
if row:
    print(requests.post(f"{LOCAL}/jobs/{row['id']}/dev-pay", params={"key": SECRET}).json())
else:
    print("no finished scans yet")

In [ ]:
# Recent jobs and the backend log
for r in db._conn().execute("SELECT id, status, n_photos, error FROM jobs ORDER BY created_at DESC LIMIT 10"):
    print(dict(r))
print(open("/content/backend.log").read()[-2000:])